# Code chien luoc => Cho chay Auto Trade (OG)
### HA PHAM gui: mcdx cắt lên 75 mua, rsi >70, macd > 0, macd > signal, macd (5,25,5), rsi (14)

### Vi du: MACD, RSI: MACD nam tren Signal_Line + RSI < 30 : Buy
### Vi du: Nguoc lai thi Sell

# Get data len

In [7]:
import sys 
sys.path.append('../Common')

import CommonMT5

# scan_marketForex()

In [8]:
import talib
import pandas as pd
import numpy as np
import datetime
import redis

def scan_marketForex():    

	# ==== Step 1: Khai bao bien============================================
	symbol = 'EURUSD.sml'
	from_date = '2025-06-01'
	to_date = '2025-08-07'
	# ==== Step 2: Load data ================================================
	data = CommonMT5.CommonMT5.loaddataMT5(symbol, from_date, to_date)

	# Processing data, tinh toan cac chi bao
	data['macd'], data['macd_signal'], data['macd_hist'] = talib.MACD(data['Close'], fastperiod=12, slowperiod=26, signalperiod=9)
	data['rsi'] = talib.RSI(data['Close'], timeperiod=14)

	# Chien luoc
	data['Buy_Signal'] = (data['macd'] > data['macd_signal']) & (data['rsi'] < 30)	
	data['Sell_Signal'] = (data['macd'] < data['macd_signal']) & (data['rsi'] > 50)

	# Kiem tra neu last_record co Buy_Signal = True hoac Sell_Signal = True thi gui qua Redis
	last_record = data.iloc[-1].copy()

	hash_key = 'OG_Bai tap 9.4_Chien luoc tai lop'
	r = redis.Redis(host='localhost', port=6379, db=0)
	
	# Chuyển đổi record cuối cùng thành từ điển và lưu vào Redis dưới dạng hash
	if(last_record['Buy_Signal'] == True or last_record['Sell_Signal'] == True):
		for field, value in last_record.to_dict().items():
			# Chuyển đổi giá trị uint64 và Timestamp thành chuỗi
			if isinstance(value, pd.Timestamp):
				value = value.isoformat()
			elif isinstance(value, (int, np.uint64)):
				value = str(value)
			r.hset(hash_key, field, value)  
			r.hset(hash_key, 'Symbol', symbol)
			r.hset(hash_key, 'Insertdate', datetime.now().isoformat())
		print(last_record)   
	else:
		print('Không có tín hiệu!')   

# Auto

In [ ]:
from datetime import datetime
import time

run_minute = list(range(0, 60, 1))
setattr(scan_marketForex, 'last_run', None)

while True:
	current_time = datetime.now()
	current_minute = current_time.minute
	current_second = current_time.second
	
	if current_minute in run_minute and current_second == 5: # Neu phut hien tai nam trong danh sach phut chay, nhung 1 phut chay hang ngan lan
		last_run = getattr(scan_marketForex, 'last_run', None)
		
		if last_run is None or last_run != current_minute:
			print(f"Chay lan {current_minute}")
			print("Chay tai thoi diem: ", current_time)
			scan_marketForex()
			setattr(scan_marketForex, 'last_run', current_minute)

	time.sleep(1)

Chay lan 40
Chay tai thoi diem:  2025-08-07 21:40:05.275499
Datetime       2025-08-06 00:00:00
Open                       1.15735
High                       1.16687
Low                        1.15644
Close                      1.16601
Volume                      126799
macd                     -0.000918
macd_signal               0.000345
macd_hist                -0.001263
rsi                      53.416201
Buy_Signal                   False
Sell_Signal                   True
Name: 47, dtype: object
Chay lan 41
Chay tai thoi diem:  2025-08-07 21:41:05.895169
Datetime       2025-08-06 00:00:00
Open                       1.15735
High                       1.16687
Low                        1.15644
Close                      1.16601
Volume                      126799
macd                     -0.000918
macd_signal               0.000345
macd_hist                -0.001263
rsi                      53.416201
Buy_Signal                   False
Sell_Signal                   True
Name: 47, dtype: